# MAE(Masked AutoEncoder) 

MAE 是一种自监督的预训练方法，一言以蔽之，就是以**一定的比例随机掩盖住**一些 patch ，然后让模型重构出完整模型的方法

<div align="center">
    <img src="./imgs/MAE_architecture.png" alt="MAE_architecture" style="width: 500px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/439554945)

[MAE 论文](https://arxiv.org/pdf/2111.06377)

[MAE Pytorch 实现](https://github.com/lucidrains/vit-pytorch/blob/main/vit_pytorch/mae.py)


## MAE 的由来

在 NLP 领域里，已经有了通过随机 mask 掉句子中的一部分让模型预测被 mask 之前的部分的内容(BERT)来让模型学习语义表示的自监督学习方法，并可以使用这种方法训练的模型进行微调，从而适配下游任务，比如预测两个句子的语义是否一致

但是因为 CV 和 NLP 之间存在 **gap**，不能直接把这种方法应用于 CV 领域中，但是当时 **ViT** 通过把图像进行 **patchify** 操作(也就是把[c, h, w]的图像转换为[seq_len, dim]的形式)，从而可以使用 **Transformer** 进行编码操作，成功把 Transformer **迁移到了图像领域**

基于这种思路，就能够对 patchify 之后的表征进行 mask(实际上就是对图像中划分的 **patch 进行掩盖操作**)，再让 AutoEncoder 编码掩盖之后的图像再解码出原图像的**自监督**训练方法，从而让模型学习图像的表征，从而应用于下游的任务


## MAE 的具体实现

### Mask 策略

MAE 的 mask 策略是：先把图像分成一个个 patch，然后使用均匀分布采样一部分 patch，剩余的 patch 全部 mask 掉(作者发现 mask 的 patch 占比在 **75** 左右是最好的，可以看下图)，然后只把没有 mask 的块输入 Encoder 中

<div align="center">
    <img src="./imgs/masking_ratio_compare.png" alt="masking_ratio_compare" style="width: 450px; height: auto;">
</div>

当然，作者也比较过不同的掩码策略，下图从左到右分别是均匀分布掩码、块状掩码和网格掩码，可以发现，均匀分布掩码策略的重建效果是最好的

<div align="center">
    <img src="./imgs/masking_strategy.png" alt="masking_strategy" style="width: 450px; height: auto;">
</div>


### Encoder & Decoder 

#### Encoder

注意，Encoder 只对**可见的非掩码 patch** 进行处理，我们可以再回顾一下怎么把图像变为 patch 序列:

首先把图像[bs, c, h, w]转换为[bs, num_patches, p\*p\*c]，其中 $p$ 表示每个 patch 的大小，然后再通过线性映射到[bs, num_patches, dim]，就变成了类似 NLP 中的[bs, seq_len, dim]的形状，从而可以使用 Transformer 进行处理了


#### Decoder

#### Decoder

对于 Decoder，则需要把掩码的 patch 重新与编码后的未掩码 patch 拼接，拼接时要按照**图像原本的位置信息进行拼接**，比如(2, 1)的 patch 没被掩码，拼接时他就还在(2, 1)的对应位置，mask 的 token 使用统一的 embedding 在潜空间进行表示(也就是所有 masked patch 的 embedding 都是一样的，对于不同位置的 masked patch，可以使用**位置编码**来进行区分)

Decoder 是在预训练的重建任务中需要的，但是下游的很多任务可能只需要 Encoder 对图像进行表示，因此 Decoder 可以不和 Encoder **对称**，可以把 Decoder 设计得更简单

<div align="center">
    <img src="./imgs/encoder-decoder.png" alt="encoder-decoder" style="width: 450px; height: auto;">
    <p><b>Encoder 与 Decoder 的不对称设计</b></p>
</div>

## MAE 代码

上面我们已经介绍完了 MAE 的核心部分，也就是掩码自监督学习和不对称的编码器-解码器设计

### 初始化

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from einops import repeat, rearrange
from c_mae import Transformer, ViT
import os
import matplotlib.pyplot as plt


class MAE(nn.Module):
    def __init__(self, *, encoder: ViT, decoder_dim, masking_ratio = 0.75, decoder_depth = 1, 
                 decoder_heads = 8, decoder_dim_head = 64):  # *表示它之后必须关键字传参
        super().__init__()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.masking_ratio = masking_ratio

        self.encoder = encoder
        num_patches, encoder_dim = encoder.pos_embedding.shape[-2:]  # [num_patches, dim]

        # Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_height, p2 = patch_width)
        self.to_patch = encoder.to_patch_embedding[0]  # -> [bs, seq_len(h*w), patch_dim]
        # nn.LayerNorm(patch_dim) -> nn.Linear(patch_dim, dim) -> nn.LayerNorm(dim)
        self.patch_to_emb = nn.Sequential(*encoder.to_patch_embedding[1:])  # -> [bs, seq_len, dim]

        pixel_values_per_patch = encoder.to_patch_embedding[2].weight.shape[-1]  # patch_dim(p1*p2*c)

        self.decoder_dim = decoder_dim
        self.enc_to_dec = nn.Linear(encoder_dim, decoder_dim) if encoder_dim != decoder_dim else nn.Identity()
        self.mask_token = nn.Parameter(torch.randn(decoder_dim))
        self.decoder = Transformer(dim = decoder_dim, depth = decoder_depth, heads = decoder_heads, dim_head = decoder_dim_head, mlp_dim = decoder_dim * 4)
        self.decoder_pos_emb = nn.Embedding(num_patches, decoder_dim)
        self.to_pixels = nn.Linear(decoder_dim, pixel_values_per_patch)  # [bs, seq_len, dim] -> [bs, seq_len, patch_dim]
        

### forward 的实现

In [2]:
def forward(self, batch_imgs:torch.Tensor):
    batch_imgs = batch_imgs.to(self.device)
    
    # -----------------------------------------------------------------------------------------
    # patchify
    # -----------------------------------------------------------------------------------------
    # [bs, 3, H, W] -> [bs, seq_len, patch_dim]
    patches = self.to_patch(batch_imgs)
    batch, num_patches, *_ = patches.shape

    # -----------------------------------------------------------------------------------------
    # Embedding，并添加位置编码
    # -----------------------------------------------------------------------------------------
    # [bs, seq_len, patch_dim] -> [bs, seq_len, dim]
    tokens = self.patch_to_emb(patches)
    if self.encoder.pool == "cls":  # 跳过cls token
        # NOTE: 这里源码报错了，源码是self.encoder.pos_embedding[:, 1:(num_patches + 1)]
        tokens += self.encoder.pos_embedding[1:(num_patches + 1)]  
    elif self.encoder.pool == "mean":
        tokens += self.encoder.pos_embedding.to(self.device, dtype=tokens.dtype)

    # -----------------------------------------------------------------------------------------
    # 按照掩码比例进行掩码操作，并编码-解码
    # -----------------------------------------------------------------------------------------
    # 获取掩码和非掩码的索引
    num_masked = int(self.masking_ratio * num_patches)
    rand_indices = torch.rand(batch, num_patches, device = self.device).argsort(dim = -1)
    masked_indices, unmasked_indices = rand_indices[:, :num_masked], rand_indices[:, num_masked:]

    # 获取不需要掩码的token进行编码
    batch_range = torch.arange(batch, device = self.device)[:, None]
    tokens = tokens[batch_range, unmasked_indices]  # -> [bs, unmasked_seq_len, dim]
    
    # 获取需要掩码的token用于计算重建损失
    masked_patches = patches[batch_range, masked_indices]
    
    # unmasked token的编码-解码
    encoded_tokens = self.encoder.transformer(tokens)
    decoder_tokens = self.enc_to_dec(encoded_tokens)
    unmasked_decoder_tokens = decoder_tokens + self.decoder_pos_emb(unmasked_indices)  # 位置还是原图像的位置

    # masked patches的编码-解码，掩码之后的token是一个统一的表示
    mask_tokens = repeat(self.mask_token, 'd -> b n d', b = batch, n = num_masked)  # -> [bs, masked_seq_len, dim]
    mask_tokens = mask_tokens + self.decoder_pos_emb(masked_indices)

    # 拼接二者
    decoder_tokens = torch.zeros(batch, num_patches, self.decoder_dim, device=self.device)
    decoder_tokens[batch_range, unmasked_indices] = unmasked_decoder_tokens
    decoder_tokens[batch_range, masked_indices] = mask_tokens
    decoded_tokens = self.decoder(decoder_tokens)

    # 取出掩码token的解码结果，并unpatchify
    mask_tokens = decoded_tokens[batch_range, masked_indices]
    # [bs, maeked_seq_len, dim] -> [bs, maeked_seq_len, patch_dim]
    pred_pixel_values = self.to_pixels(mask_tokens)

    # -----------------------------------------------------------------------------------------
    # 计算重建损失
    # -----------------------------------------------------------------------------------------
    recon_loss = F.mse_loss(pred_pixel_values, masked_patches)

    return recon_loss


MAE.forward = forward


In [3]:
from c_mae import get_config, get_vit


# --------------------------------------------------------------------------------------
# constants
# --------------------------------------------------------------------------------------
IMG_SIZE = 64
BATCH_SIZE = 4
PATCH_SIZE = 16


config = get_config("./configs/vit_mae_config.yaml")
vit = get_vit(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mae = MAE(
    encoder=vit,
    decoder_dim=256,
    masking_ratio=0.75,
    decoder_depth=1,
    decoder_heads=8,
    decoder_dim_head=64
).to(device)

imgs = torch.randint(0, 256, (BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)).to(device).float()
print(f'Img size: {imgs.size()}')
print(f"Each patch size: {PATCH_SIZE}")
loss = mae(imgs)
print(f'Loss: {loss}')


Img size: torch.Size([4, 3, 64, 64])
Each patch size: 16
Loss: 21682.193359375


### 可视化 masked image

In [4]:
@torch.no_grad()
def vis_mask_and_unmask(self, batch_imgs:torch.Tensor, save_dir="./imgs/mask_vis"):
    self.eval()
    batch = batch_imgs.shape[0]
    os.makedirs(save_dir, exist_ok=True)
    batch_imgs = batch_imgs.to(self.device)

    patches = self.to_patch(batch_imgs)
    batch, num_patches, *_ = patches.shape

    num_masked = int(self.masking_ratio * num_patches)
    rand_indices = torch.rand(batch, num_patches, device = self.device).argsort(dim = -1)
    masked_indices = rand_indices[:, :num_masked]

    masked_patches = patches.clone()
    batch_idx = torch.arange(batch, device=self.device)[:, None]
    masked_patches[batch_idx, masked_indices] = 0.5

    *_, image_height, _ = batch_imgs.shape

    patches_to_img = lambda p: rearrange(
        p, 
        'b (h w) (p1 p2 c) -> b c (h p1) (w p2)',
        h=image_height // self.encoder.patch_height,
        p1=self.encoder.patch_height,
        p2=self.encoder.patch_width,
        c=3
    )

    img_original = patches_to_img(patches)
    img_masked = patches_to_img(masked_patches)

    for i in range(batch):
        ori = img_original[i].cpu().permute(1, 2, 0).numpy()
        msk = img_masked[i].cpu().permute(1, 2, 0).numpy()

        fig, axes = plt.subplots(1, 2, figsize=(5, 3))
        fig.subplots_adjust(wspace=0.3)

        axes[0].imshow(ori)
        axes[0].set_title("origin")
        axes[0].axis('off')

        axes[1].imshow(msk)
        axes[1].set_title("masked")
        axes[1].axis('off')

        save_path = os.path.join(save_dir, f"mask_and_unmask_vis_{i}.png")
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
        plt.close(fig)


MAE.vis_mask_and_unmask = vis_mask_and_unmask


In [5]:
from torchvision import transforms
from PIL import Image


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

img_paths = []
for cls in ["0", "1"]:
    folder = f"./train/{cls}"
    for f in os.listdir(folder):
        if f.endswith(".png"):
            img_paths.append(os.path.join(folder, f))

imgs = []
for p in img_paths:
    img = Image.open(p).convert("RGB")
    img = transform(img)
    imgs.append(img)
imgs = torch.stack(imgs)

mae.vis_mask_and_unmask(
    batch_imgs=imgs,
    save_dir="./imgs/mask_vis"
)


<div align="center">
    <img src="./imgs/mask_vis/mask_and_unmask_vis_0.png" alt="mask_and_unmask_vis_0" style="width: 300px; height: auto;">
    <p><b>示例1</b></p>
</div>

<div align="center">
    <img src="./imgs/mask_vis/mask_and_unmask_vis_1.png" alt="mask_and_unmask_vis_1" style="width: 300px; height: auto;">
    <p><b>示例2</b></p>
</div>

### 预训练 MAE 示例

In [7]:
from c_mae import get_mae, train_mae


config = get_config("./configs/vit_mae_config.yaml")
pretrain_config = config["pretrain"]
mae = get_mae(config)

train_mae(
    mae_model=mae,
    img_size=config["global"]["img_size"],
    checkpoint_save_dir=pretrain_config["checkpoint_save_dir"],
    batch_size=pretrain_config["batch_size"],
    lr=pretrain_config["lr"],
    epochs=pretrain_config["epochs"],
    dataset_dir=pretrain_config["train_dir"]
)


Training MAE: 100%|██████████| 100/100 [00:04<00:00, 23.89it/s, avg_loss=0.0231]


[trainer]Saved: vit_iter100.pth


### 微调 MAE Encoder 适配图像分类任务

In [8]:
from c_mae import train_vit_finetune


config = get_config("./configs/vit_mae_config.yaml")
finetune_config = config["fine_tune"]
vit = get_vit(config, load_pretrained=[True, "pretrained"])

train_vit_finetune(
    vit_model=vit,
    img_size=config["global"]["img_size"],
    checkpoint_save_dir=finetune_config["checkpoint_save_dir"],
    batch_size=finetune_config["batch_size"],
    lr=finetune_config["lr"],
    epochs=finetune_config["epochs"],
    dataset_dir=finetune_config["train_dir"]
)


[ViT]Loaded checkpoint from ./ckpts/pretrained/vit_iter5000.pth


Fine-tuning ViT: 100%|██████████| 50/50 [00:01<00:00, 25.66it/s, avg_loss=0.0592, acc=1]


[trainer]Saved: vit_finetune_iter50.pth


In [9]:
config = get_config("./configs/vit_mae_config.yaml")
vit = get_vit(config, load_pretrained=[True, "finetuned"])
class_idx_to_name = {
    0: "高松灯",
    1: "菲比"
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vit.to(device)
vit.eval()

transform = transforms.Compose([
    transforms.Resize((config["vit"]["image_size"], config["vit"]["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

test_imgs = []
true_labels = []
for cls in ["0", "1"]:
    folder = f"./train/{cls}"
    for f in os.listdir(folder):
        if f.endswith(".png"):
            img_path = os.path.join(folder, f)
            img = Image.open(img_path).convert("RGB")
            img = transform(img)
            test_imgs.append(img)
            true_labels.append(int(cls))

test_imgs = torch.stack(test_imgs).to(device)

with torch.no_grad():
    logits = vit(test_imgs)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

for i, (pred, true) in enumerate(zip(preds, true_labels)):
    print(f"img{i}: true label = {class_idx_to_name[true]}, pred label = {class_idx_to_name[pred]}, result = {true == pred}")
    print('-'*114)


[ViT]Loaded checkpoint from ./ckpts/pretrained/vit_finetune_iter50.pth
img0: true label = 高松灯, pred label = 高松灯, result = True
------------------------------------------------------------------------------------------------------------------
img1: true label = 菲比, pred label = 菲比, result = True
------------------------------------------------------------------------------------------------------------------
